In [0]:
%run ./00_config

In [0]:
%sql
-- Garante que conseguimos ler a Silver antes de fazer os cálculos
CREATE SCHEMA IF NOT EXISTS hive_metastore.crypto_silver
LOCATION 'abfss://silver@stcryptolakehouse.dfs.core.windows.net/';

CREATE EXTERNAL TABLE IF NOT EXISTS hive_metastore.crypto_silver.crypto_prices_clean
USING DELTA
LOCATION 'abfss://silver@stcryptolakehouse.dfs.core.windows.net/crypto_prices_clean';

In [0]:
from pyspark.sql.functions import col, to_date, avg, max, min, sum, current_timestamp, round

df_silver = spark.read.table("hive_metastore.crypto_silver.crypto_prices_clean")

df_gold = df_silver \
    .withColumn("data_referencia", to_date(col("data_hora_cotacao"))) \
    .groupBy("moeda_id", "data_referencia") \
    .agg(
        round(avg("preco_usd"), 2).alias("preco_medio_usd"),
        round(max("preco_usd"), 2).alias("preco_maximo_usd"),
        round(min("preco_usd"), 2).alias("preco_minimo_usd"),
        round(sum("volume_24h"), 2).alias("volume_total_dia")
    ) \
    .withColumn("timestamp_processamento_gold", current_timestamp()) \
    .orderBy(col("data_referencia").desc(), col("moeda_id"))

caminho_gold = "abfss://gold@stcryptolakehouse.dfs.core.windows.net/crypto_daily_metrics"
df_gold.write.format("delta").mode("overwrite").save(caminho_gold)

In [0]:
%sql
-- Salva a métrica agregada no catálogo final
CREATE SCHEMA IF NOT EXISTS hive_metastore.crypto_gold
LOCATION 'abfss://gold@stcryptolakehouse.dfs.core.windows.net/';

CREATE EXTERNAL TABLE IF NOT EXISTS hive_metastore.crypto_gold.crypto_daily_metrics
USING DELTA
LOCATION 'abfss://gold@stcryptolakehouse.dfs.core.windows.net/crypto_daily_metrics';